In [21]:
from pyspark.sql import SparkSession
import getpass 
username=getpass.getuser()
spark=SparkSession. \
    builder. \
    config('spark.ui.port','0'). \
    config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    master('yarn'). \
    getOrCreate()

In [22]:
orders_schema = "order_id long, order_date date, customer_id long, order_status string"

In [23]:
orders_schema

'order_id long, order_date date, customer_id long, order_status string'

In [24]:
df = spark.read \
.format("csv") \
.schema(orders_schema) \
.option("dateFormat", "MM-dd-yyyy") \
.load("/public/trendytech/datasets/orders_sample2.csv")

In [25]:
df.show()

+--------+----------+-----------+---------------+
|order_id|order_date|customer_id|   order_status|
+--------+----------+-----------+---------------+
|       1|2013-07-25|      11599|         CLOSED|
|       2|2013-07-25|        256|PENDING_PAYMENT|
|       3|2013-07-25|      12111|       COMPLETE|
|       4|2013-07-25|       8827|         CLOSED|
|       5|2013-07-25|      11318|       COMPLETE|
|       6|2013-07-25|       7130|       COMPLETE|
|       7|2013-07-25|       4530|       COMPLETE|
|       8|2013-07-25|       2911|     PROCESSING|
|       9|2013-07-25|       5657|PENDING_PAYMENT|
|      10|2013-07-25|       5648|PENDING_PAYMENT|
+--------+----------+-----------+---------------+



In [26]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [27]:
from pyspark.sql.functions import *

In [35]:
orders_schema = "order_id long, order_date string, customer_id long, order_status string"

In [29]:
df = spark.read \
.format("csv") \
.schema(orders_schema) \
.load("/public/trendytech/datasets/orders_sample2.csv")

In [30]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [31]:
new_df = df.withColumn("orders_date_new", to_date("order_date", "MM-dd-yyyy"))

In [32]:
new_df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- orders_date_new: date (nullable = true)



In [33]:
new_df.show()

+--------+----------+-----------+---------------+---------------+
|order_id|order_date|customer_id|   order_status|orders_date_new|
+--------+----------+-----------+---------------+---------------+
|       1|07-25-2013|      11599|         CLOSED|     2013-07-25|
|       2|07-25-2013|        256|PENDING_PAYMENT|     2013-07-25|
|       3|07-25-2013|      12111|       COMPLETE|     2013-07-25|
|       4|07-25-2013|       8827|         CLOSED|     2013-07-25|
|       5|07-25-2013|      11318|       COMPLETE|     2013-07-25|
|       6|07-25-2013|       7130|       COMPLETE|     2013-07-25|
|       7|07-25-2013|       4530|       COMPLETE|     2013-07-25|
|       8|07-25-2013|       2911|     PROCESSING|     2013-07-25|
|       9|07-25-2013|       5657|PENDING_PAYMENT|     2013-07-25|
|      10|07-25-2013|       5648|PENDING_PAYMENT|     2013-07-25|
+--------+----------+-----------+---------------+---------------+



## READ MODES IN SPARK

#### FAILFAST, DROPMALFROMED, PERMISSIVE(DEFAULT)

In [42]:
#failfast-- error out if malformed data found
#permissive- if mismatch record found it will give null instead of error
#dropmalformed-- drops the records 

df = spark.read \
.format("csv") \
.schema(orders_schema) \
.option("mode", "dropmalformed") \
.load("/public/trendytech/datasets/orders_sample3.csv")

In [43]:
df.show()

+--------+----------+-----------+---------------+
|order_id|order_date|customer_id|   order_status|
+--------+----------+-----------+---------------+
|       1|2013-07-25|      11599|         CLOSED|
|       2|2013-07-25|        256|PENDING_PAYMENT|
|       3|2013-07-25|      12111|       COMPLETE|
|       4|2013-07-25|       8827|         CLOSED|
|       5|2013-07-25|      11318|       COMPLETE|
|       6|2013-07-25|       7130|       COMPLETE|
|       8|2013-07-25|       2911|     PROCESSING|
|      10|2013-07-25|       5648|PENDING_PAYMENT|
+--------+----------+-----------+---------------+



In [44]:
#failfast-- error out if malformed data found
#permissive- if mismatch record found it will give null instead of error
#dropmalformed-- drops the records 

df = spark.read \
.format("csv") \
.schema(orders_schema) \
.option("mode", "permissive") \
.load("/public/trendytech/datasets/orders_sample3.csv")

In [45]:
df.show()

+--------+----------+-----------+---------------+
|order_id|order_date|customer_id|   order_status|
+--------+----------+-----------+---------------+
|       1|2013-07-25|      11599|         CLOSED|
|       2|2013-07-25|        256|PENDING_PAYMENT|
|       3|2013-07-25|      12111|       COMPLETE|
|       4|2013-07-25|       8827|         CLOSED|
|       5|2013-07-25|      11318|       COMPLETE|
|       6|2013-07-25|       7130|       COMPLETE|
|       7|2013-07-25|       null|       COMPLETE|
|       8|2013-07-25|       2911|     PROCESSING|
|       9|2013-07-25|       null|PENDING_PAYMENT|
|      10|2013-07-25|       5648|PENDING_PAYMENT|
+--------+----------+-----------+---------------+



In [46]:
#failfast-- error out if malformed data found
#permissive- if mismatch record found it will give null instead of error
#dropmalformed-- drops the records 

df = spark.read \
.format("csv") \
.schema(orders_schema) \
.option("mode", "failfast") \
.load("/public/trendytech/datasets/orders_sample3.csv")